# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

To find out, whether there is attenuated behaviour of opioids drugs, by conparing the expression of genes in different brain regions

What do the conditions mean?

oxy: The opioids drug Oxycodone is applied to he mice.


sal: Control group of "oxy", the opioids is absent.

What do the genotypes mean?

SNI: spared nerve injury

Sham: Control group of SNI, the procedure of causing the dadmage is conducted, without doing any real damage.

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

1. What would you do?

2. Which groups would you compare to each other?

3. Please also mention which outcome you would expect to see from each comparison.

1. Compare the gene expression data of 4 groups
 
2-a. SNI-oxy vs. SNI-sal, to compare how would withdrawl of the drug impact expression level.
2-b. first comapre, Sham-oxy vs. Sham-sal, and compare the result to the result of step a, to make sure the effect is caused by withdral instead of pain.

3. The expression level should be different in comparison a, cause SNI-oxy really has a withdrawl exposure while the other group doesn't. The comparison between the sham mice should als have similar difference, maybe a little weaker, since the opioid isn't really needed there. 


Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [4]:
import pandas as pd

df = pd.read_excel("conditions_runs_oxy_project.xlsx")
df.head(5)

,Patient,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham
0,?,SRR23195505,x,NaN,x,NaN,x,NaN
1,?,SRR23195506,x,NaN,NaN,x,NaN,x
2,?,SRR23195507,x,NaN,x,NaN,NaN,x
3,?,SRR23195508,x,NaN,NaN,x,x,NaN
4,?,SRR23195509,x,NaN,NaN,x,x,NaN


In [5]:
df.count(axis=0,numeric_only=False)

Patient           16
Run               16
RNA-seq           16
DNA-seq            0
condition: Sal     8
Condition: Oxy     8
Genotype: SNI      8
Genotype: Sham     8
dtype: int64

1.There are 8 samples per conddition\
2.There are 8 samples per genotype
3. There are 4 fr each combination

In [7]:
import numpy as np

df.columns = df.columns.str.strip()   # remove stray spaces in column names

tidy = pd.DataFrame({
    "run": df["Run"].astype(str).str.strip(),
    "condition": np.where(df["Condition: Oxy"].notna(), "Oxy", "Sal"),
    "genotype": np.where(df["Genotype: SNI"].notna(), "SNI", "Sham"),
}).sort_values("run").reset_index(drop=True)

tidy

,run,condition,genotype
0,SRR23195505,Sal,SNI
1,SRR23195506,Oxy,Sham
2,SRR23195507,Sal,Sham
3,SRR23195508,Oxy,SNI
4,SRR23195509,Oxy,SNI
5,SRR23195510,Sal,SNI
6,SRR23195511,Oxy,Sham
7,SRR23195512,Sal,Sham
8,SRR23195513,Sal,SNI
9,SRR23195514,Oxy,Sham


In [8]:
tidy["condition"].value_counts()                    # 1. per condition
tidy["genotype"].value_counts()                     # 2. per genotype
pd.crosstab(tidy["genotype"], tidy["condition"])    # 3. condition per genotype

condition,Oxy,Sal
genotype,,
SNI,4,4
Sham,4,4


They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [9]:
# lload base count to the table

bases = pd.read_csv("base_counts.csv")
bases["Run"] = bases["Run"].str.strip()

tidy = tidy.merge(bases, left_on="run", right_on="Run", how="left").drop(columns="Run")
tidy = tidy.rename(columns={"Bases": "bases"})
tidy

,run,condition,genotype,bases
0,SRR23195505,Sal,SNI,6922564500
1,SRR23195506,Oxy,Sham,7859530800
2,SRR23195507,Sal,Sham,8063298900
3,SRR23195508,Oxy,SNI,6927786900
4,SRR23195509,Oxy,SNI,7003550100
5,SRR23195510,Sal,SNI,7377388500
6,SRR23195511,Oxy,Sham,6456390900
7,SRR23195512,Sal,Sham,7462857900
8,SRR23195513,Sal,SNI,8099181600
9,SRR23195514,Oxy,Sham,7226808600


In [11]:
# sort

tidy = tidy.sort_values(["condition","genotype", "bases"]).reset_index(drop=True)
tidy

,run,condition,genotype,bases
0,SRR23195516,Oxy,SNI,6203117700
1,SRR23195517,Oxy,SNI,6863840400
2,SRR23195508,Oxy,SNI,6927786900
3,SRR23195509,Oxy,SNI,7003550100
4,SRR23195511,Oxy,Sham,6456390900
5,SRR23195519,Oxy,Sham,6996050100
6,SRR23195514,Oxy,Sham,7226808600
7,SRR23195506,Oxy,Sham,7859530800
8,SRR23195505,Sal,SNI,6922564500
9,SRR23195510,Sal,SNI,7377388500


In [12]:
# select 2 smallest run

smallest = tidy.nsmallest(2, "bases")
smallest

,run,condition,genotype,bases
0,SRR23195516,Oxy,SNI,6203117700
4,SRR23195511,Oxy,Sham,6456390900


In [13]:
# save the id to a file for the pipeline
smallest["run"].to_csv("ids.csv", index=False, header=False)

In [15]:
!nextflow run nf-core/fetchngs -r 1.13.0 -profile docker --input ids.csv --outdir results/fetchngs


 N E X T F L O W   ~  version 26.04.6

Pulling nf-core/fetchngs:1.13.0 ...
 downloaded from https://github.com/nf-core/fetchngs.git
Launching `https://github.com/nf-core/fetchngs` [curious_coulomb] revision: 955c892d80 [1.13.0]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/fetchngs 1.13.0
------------------------------------------------------

Input/output options
  input              : ids.csv
  outdir             : results/fetchngs

Deprecated options
  trace_report_suffix: 2026-09-29_11-31-45

Core Nextflow options
  revision           : 1.13.0
  runName            : curious_coulomb
  containerEngine    : docker
  launchDir          : /Users/ivan/Desktop/CompWorkflow/co

The downloading failed, further steps will be continued from copied results.

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

How I would try to reproduce the analysis:
1. get all the data and metadata
2. quality control, preprocessing nf-coore/rnaseq
3. Differential expression: nf-core/differentialabundance
4. Downstream analysis